# Visualization: Forecast, Leaderboard, Diagnostics
# 可视化：预测、排行榜与诊断

Scenario: analysts need reusable visual checks for demand forecasts before sending results to planning systems.

场景：分析师需要在预测结果进入计划系统前进行可复用的可视化检查。

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
from PipelineTS.pipeline import ModelPipeline
from PipelineTS.plot import (
    TSPlotter,
    plot_series,
    plot_forecast,
    plot_leaderboard,
    plot_leaderboard_detail,
    plot_model_comparison,
    plot_residuals,
    plot_acf_pacf,
    plot_decomposition,
    plot_train_test_split,
)

data = make_retail_demand(n_days=220, n_stores=1).drop(columns=["store_id"])
train, valid = data.iloc[:-21].copy(), data.iloc[-21:].copy()

In [ ]:
plot_series(data, time_col="date", target_col="sales", title="Retail demand history", lang="zh")
plot_train_test_split(train, valid, time_col="date", target_col="sales", lang="zh")
plot_decomposition(train, time_col="date", target_col="sales", lang="zh")
plot_acf_pacf(train["sales"].values, max_lags=30, lang="zh")

In [ ]:
pipe = ModelPipeline(
    time_col="date",
    target_col="sales",
    lags=14,
    include_models=["random_forest", "extra_forest", "multi_output_model"],
    quantile=0.9,
    cv=2,
    random_forest__n_estimators=80,
    extra_forest__n_estimators=80,
)
leaderboard = pipe.fit(train, valid_data=valid)
pred = pipe.predict(21)

In [ ]:
plot_forecast(train, pred, time_col="date", target_col="sales", history_tail=90, lang="zh")
plot_leaderboard(leaderboard, lang="zh")
plot_leaderboard_detail(leaderboard, lang="zh")

In [ ]:
predictions = {
    name: pipe.predict(21, model_name=name)
    for name in pipe.leader_board_["model"].head(3)
}
plot_model_comparison(train, predictions, time_col="date", target_col="sales", history_tail=90, lang="zh")

In [ ]:
y_true = valid["sales"].values[:len(pred)]
y_pred = pred["sales"].values[:len(y_true)]
plot_residuals(y_true, y_pred, time_index=valid["date"].values[:len(y_true)], lang="zh")

In [ ]:
plotter = TSPlotter(time_col="date", target_col="sales", lang="zh")
plotter.plot_series(data)
plotter.plot_forecast(train, pred, history_tail=60)
plotter.plot_leaderboard(pipe.leader_board_)